In [1]:
import subprocess
import sys
import re
import os
import pandas as pd
from matplotlib import pyplot as plt
import matplotlib as mpl
import seaborn as sns
import seaborn.objects as so
import warnings
warnings.filterwarnings("ignore")
import multiprocessing as mp

from astropy import units as u
import numpy as np
from astropy.constants import G, sigma_sb, k_B

from scipy.interpolate import PchipInterpolator
from process_evotracks import evolution_track as track

In [2]:

E_imp = 0

prefix_005 = "./PlanetSolver -cn 1 0.0162495 -en 2 0.6412495 -a 9.50 30 0.05 -s 1 1 1 0.3 -m 8.0 -evolve 1"
prefix_01 = "./PlanetSolver -cn 1 0.2925 -en 2 0.60749 -a 8.9 30 0.0999 -s 1 1 1 0.3 -m 8.0 -evolve 1"
prefix_02 = "./PlanetSolver -cn 1 0.2599 -en 2 0.53995 -a 8.38 30 0.199 -s 1 1 1 0.3 -m 8.0 -evolve 1"
prefix_03 = "./PlanetSolver -cn 1 0.227495 -en 2 0.47249  -a 7.96 30 0.299 -s 1 1 1 0.3 -m 8.0 -evolve 1"

prefix_dict = {0.1:prefix_01, 0.2:prefix_02, 0.3:prefix_03}

runswitch = True
compilation = True #Re-compile every time
AMF = 0.1

controlEvoFile = f"controls/evo-control-f{str(AMF).replace(".","")}.dat"

structFileName = "/dne/non_esiste.dat" #f"sim_results/impact-fenv{str(AMF).replace(".","")}-struct.dat"
evolveFileName = f"sim_results/impact-fenv{str(AMF).replace(".","")}-evo.dat"


if runswitch and compilation:
    result = subprocess.run(
    "make clean", 
    shell=True, 
    capture_output=True, 
    text=True,
    timeout=300
    )
    if result.returncode != 0:
        raise OSError("OS could not handle make clean")
    print("Cleaned")

    result = subprocess.run(
    "make", 
    shell=True, 
    capture_output=True, 
    text=True,
    timeout=300
    )
    if result.returncode != 0:
        print(result.stdout)
        raise OSError(f"Could not make:{result.stderr}")
    print("Made")



Cleaned
Made


In [3]:

def get_allfiles(directory='sim_results', massfrac=0.1):
    """
    Create a dictionary mapping float values (0.Y) to filenames.
    
    Args:
        directory: Path to the directory containing the files (default: current directory)
    
    Returns:
        dict[float, str]: Dictionary mapping float values to filenames
    """
    result = {}
    massfrac_str = str(massfrac).replace(".", "")
    # Pattern to match impact-fenvXX-evo-0-Y.dat
    pattern = rf'impact-fenv{massfrac_str}-evo-0-(\d+)\.dat'
    
    # Iterate through files in directory
    for filename in os.listdir(directory):
        match = re.match(pattern, filename)
        if match:
            # Extract Y value
            y_value = match.group(1)
            # Create float as 0.Y
            key = float(f"0.{y_value}")
            result[key] = filename
    
    return result



In [4]:
#multiprocessing
low_age = []
exception_raised = []
short_files = []
def check_sims(impact_mass_fraction, impact_time = 1000):
    
    assert impact_time % 1 == 0, "Don't input decimal impact times"

    #ASSUMPTIONS TO BE CHECKED AFTER TESTING
    M_p = 8 * u.Mearth
    R_p = 10 * u.Rearth
    f_env = AMF #TODO change this every time the f_env is different
    M_core = (1-f_env) * M_p
    R_c = R_p * (M_core/M_p) ** 4  
    eta = 0.5 # Realistic ish value
    

    M_imp = impact_mass_fraction * M_core
    E_imp = eta * M_imp * (G * M_p)/R_c
 
    pcmd = prefix_dict[AMF]
    magstr = str(impact_mass_fraction).replace(".","-")
    #filestr = f"{evolveFileName[:-4]}-{magstr}.dat" #for OG impact sims 
    filestr = f"{evolveFileName[:-4]}-{magstr[0:6]}.dat"
    try:
        


        try:
            with open(f"{filestr}", "r") as file:
                lines = sum(1 for line in file)
                if lines < 200: 
                    short_files.append(impact_mass_fraction)
                    print(f"Error {filestr} - short file")
                    
               
            
        
            df = track.load_track(f"{filestr}")
            if max(df["Age"]) < 8000:
                low_age.append(impact_mass_fraction)
                print(f"Error {filestr} - low age")
                
                
               
        except FileNotFoundError as fne:
            print(fne.with_traceback(None))
            print(f"File does not exist: {f"{filestr}.dat"}")
             #increase offset to maybe prevent stiff equations?
            
            
            print(f"error fne: {impact_mass_fraction}, {f_env} -> {filestr}.dat")
            

        except Exception as e:
            print(e.with_traceback(None))
            exception_raised.append(impact_mass_fraction)
            return f"Failed: {impact_mass_fraction} -> {filestr}"
        #return result
    except subprocess.TimeoutExpired as timeout:
        print(timeout.with_traceback(None))
        return None
#impact_massfracs = [0.0080001, 0.00801, 0.0080000001, 0.00800001, 0.008] 
#impact_massfracs = np.linspace(0.001, 0.009, 50) #fenv03 
impact_massfracs = get_allfiles(massfrac=AMF).keys() #fenv01 and 02
impact_time = 1000 #Myr
for mf in impact_massfracs:
    check_sims(mf)


Error sim_results/impact-fenv01-evo-0-0055.dat - low age
Error sim_results/impact-fenv01-evo-0-0057.dat - low age
Error sim_results/impact-fenv01-evo-0-006.dat - short file
Error sim_results/impact-fenv01-evo-0-006.dat - low age
Error sim_results/impact-fenv01-evo-0-0062.dat - short file
Error sim_results/impact-fenv01-evo-0-0062.dat - low age
Error sim_results/impact-fenv01-evo-0-0065.dat - short file
Error sim_results/impact-fenv01-evo-0-0065.dat - low age
Error sim_results/impact-fenv01-evo-0-0067.dat - short file
Error sim_results/impact-fenv01-evo-0-0067.dat - low age
Error sim_results/impact-fenv01-evo-0-007.dat - short file
Error sim_results/impact-fenv01-evo-0-007.dat - low age
Error tokenizing data. C error: Expected 7 fields in line 144, saw 8



In [5]:
low_age

[0.0055, 0.0057, 0.006, 0.0062, 0.0065, 0.0067, 0.007]

In [6]:
short_files

[0.006, 0.0062, 0.0065, 0.0067, 0.007]

In [7]:
impact_massfracs

dict_keys([0.0012, 0.001, 0.0015, 0.0017, 0.0025, 0.0022, 0.002, 0.0027, 0.003, 0.0032, 0.0035, 0.0037, 0.004, 0.0042, 0.0045, 0.0047, 0.005, 0.0052, 0.0055, 0.0057, 0.006, 0.0062, 0.0065, 0.0067, 0.007, 0.0003, 0.0005, 0.0006, 0.0007, 0.0001, 0.0009])

In [8]:
#multiprocessing
debug=False
if debug:
    runswitch = True
def run_sims(impact_mass_fraction, impact_time = 1000):
    
    assert impact_time % 1 == 0, "Don't input decimal impact times"

    #ASSUMPTIONS TO BE CHECKED AFTER TESTING
    M_p = 8 * u.Mearth
    R_p = 10 * u.Rearth
    f_env = AMF #TODO change this every time the f_env is different
    M_core = (1-f_env) * M_p
    R_c = R_p * (M_core/M_p) ** 4  
    eta = 0.5 # Realistic ish value
    

    M_imp = impact_mass_fraction * M_core
    E_imp = eta * M_imp * (G * M_p)/R_c
 
    pcmd = prefix_dict[AMF]
    magstr = str(impact_mass_fraction).replace(".","-")
    filestr = f"sim_results/{get_allfiles(massfrac=AMF)[impact_mass_fraction]}"
    magstr = str(impact_mass_fraction).replace(".","-")
    runcmd = (pcmd
            + f" -impact {str(E_imp.si.value).replace("+", "")} {int(impact_time)} -struct {structFileName[:-4]}-{magstr}.dat -wr {filestr}")
    print(f"run: {impact_mass_fraction}, {f_env} -> {filestr}\n")
    
    if debug:
        print(f"({magstr}, {AMF}, {filestr}):\n{runcmd}\n")
        return None
    try:
        
        
        result = subprocess.run(
        runcmd, 
        shell=True, 
        capture_output=True, 
        text=True,
        timeout=900
        )
    
        if result.returncode != 0:
            print(result.stdout)
            raise OSError("Runtime error")
        print(f"Run Success: {impact_mass_fraction}, {f_env} -> {filestr}\n")

        
        return result
    except subprocess.TimeoutExpired as timeout:
        print(timeout.with_traceback(None))
        return None
#impact_massfracs = [0.0080001, 0.00801, 0.0080000001, 0.00800001, 0.008]

impact_time = 1000 #Myr
if runswitch:
    with mp.Pool(processes=mp.cpu_count()-1) as pool:
        all_results = pool.map(run_sims, low_age)

run: 0.007, 0.1 -> sim_results/impact-fenv01-evo-0-007.dat
run: 0.0055, 0.1 -> sim_results/impact-fenv01-evo-0-0055.dat
run: 0.0067, 0.1 -> sim_results/impact-fenv01-evo-0-0067.dat
run: 0.006, 0.1 -> sim_results/impact-fenv01-evo-0-006.dat
run: 0.0057, 0.1 -> sim_results/impact-fenv01-evo-0-0057.dat


run: 0.0062, 0.1 -> sim_results/impact-fenv01-evo-0-0062.dat
run: 0.0065, 0.1 -> sim_results/impact-fenv01-evo-0-0065.dat







Command './PlanetSolver -cn 1 0.2925 -en 2 0.60749 -a 8.9 30 0.0999 -s 1 1 1 0.3 -m 8.0 -evolve 1 -impact 9.338469792041252e30 1000 -struct /dne/non_esiste-0-0057.dat -wr sim_results/impact-fenv01-evo-0-0057.dat' timed out after 900 secondsCommand './PlanetSolver -cn 1 0.2925 -en 2 0.60749 -a 8.9 30 0.0999 -s 1 1 1 0.3 -m 8.0 -evolve 1 -impact 1.064913221899441e31 1000 -struct /dne/non_esiste-0-0065.dat -wr sim_results/impact-fenv01-evo-0-0065.dat' timed out after 900 seconds

Command './PlanetSolver -cn 1 0.2925 -en 2 0.60749 -a 8.9 30 0.0999 -s 1 1 1 0.3 -m 8.0 -evolve 1 -impact 1.1468296235840135e31 1000 -struct /dne/non_esiste-0-007.dat -wr sim_results/impact-fenv01-evo-0-007.dat' timed out after 900 seconds
Command './PlanetSolver -cn 1 0.2925 -en 2 0.60749 -a 8.9 30 0.0999 -s 1 1 1 0.3 -m 8.0 -evolve 1 -impact 9.829968202148686e30 1000 -struct /dne/non_esiste-0-006.dat -wr sim_results/impact-fenv01-evo-0-006.dat' timed out after 900 seconds
Command './PlanetSolver -cn 1 0.2925 -e

In [9]:
low_age

[0.0055, 0.0057, 0.006, 0.0062, 0.0065, 0.0067, 0.007]